In [ ]:
import jax
import numpy as np
from master import *
from collections import defaultdict
import pickle

jax.config.update("jax_enable_x64", True)
jax.devices()
device_id = 1
device.setup_device(device_id=device_id)

In [ ]:
PTYCHOGRAPHY = True

if PTYCHOGRAPHY:
    init_algo = "rPIE"
    # base_path = "/data/multispot_pty/pty-64-final/"
    base_path = "/data/multispot_pty/pty-64-jitter-full/"
    # base_path = "/data/multispot_pty/pty-64-jitter-10k/"
    fig_path = "../docs/thesis/figures/pty"
    est_id = "rPIE-second-order-ml"
    est_label = "rPIE+ML"
    stats_pickle_path = "MSE-dict-pty.pickle"
    bounds_pickle_path = "MSE-dict-pty-bounds.pickle"
else:
    init_algo = "HIO"
    base_path = "cdi-128/"
    fig_path = "../docs/thesis/figures/cdi"
    est_id = "ORACLE-averaged-second-order-ml"
    est_label = "ORACLE+FS"
    stats_pickle_path = "MSE-dict-v2.pickle"
    bounds_pickle_path = "MSE-dict-cdi-bounds.pickle"

loader = Loader(base_path)
SAVE_FIGS = True

In [ ]:
from master.spotgen import create_multispot_array_product_jittered
from pty_defs import ProbeParams, PtySim

pixel_intensity = 1e10


meas_id = f"CDI-intensity-{pixel_intensity:.3g}"
x_true_abs, d, probes, _, _, measurement_params, measurement_hash_meas = (
    loader.get_measurement_with_probes(meas_id)
)
x_est, simulation_params, lls, x_est_raw, config_hash, measurement_hash_est = (
    loader.get_estimate_with_x_est_raw(meas_id, est_id)
)
# Check that measurement corresponds to reconstruction
assert measurement_hash_est == measurement_hash_meas
sim_params = simulation_params["simulation"]

N_crop = sim_params["N_crop"]
N_pad = sim_params["N_pad"]
N_photons = sim_params["N_photons"]
N_samples = sim_params["N_samples"]

if PTYCHOGRAPHY:
    probe_dict = simulation_params["probe"]
    M = sim_params["M"]
    NA_det = sim_params["NA_det"]
    probe_params = ProbeParams.from_dict(probe_dict)
    print(probe_params)
    print(NA_det)
    pty_sim = PtySim(N_crop, N_pad, NA_det)
    x, y, X, Y, R, k_0, k_x, k_y, K_X, K_Y, k, r_extent, k_extent = (
        pty_sim.create_axes()
    )
    obj_support = pty_sim.obj_support
    A, A_dagger, F_pupil, pupil_support = pty_sim.get_ops_and_pupil()

    probes_for_support, N_photons, N_coeff = (
        pty_sim.create_probe_with_average_intensity(
            M,
            probe_params,
            1e6,
            f_probe=create_multispot_array_product_jittered,
            delta_r_coeff=0.50,
            use_shift=True,
            r_frac_jitter=0.1,
        )
    )
    _, spectral_support, P_spectral_support = pty_sim.construct_spectral_support_rel(
        probes_for_support[0]
    )
    x_true = P_spectral_support(x_true_abs)
else:
    x, y, X, Y, R, k_0, k_x, k_y, K_X, K_Y, k, r_extent, k_extent = (
        create_physical_axes(x_true_abs, pixel_size=1)
    )
    r_extent_cropped = list(
        create_physical_axes(center_crop(x_true_abs, N_crop), pixel_size=1)
    )[-2]
    x_true = x_true_abs
    print(k_extent)

N_samples = x_est.shape[0]
N_samples, x_true_abs.shape, d.shape, x_est.shape, simulation_params["simulation"]

In [ ]:
plot_intensity_phase(x_true, extent=r_extent)
plot_intensity_phase(fft2_shifted(x_true), extent=r_extent, use_log=True)

# MSE for all estimates and $N_{photon}$ values 

In [ ]:
from master.estimator import *

print(x_est.shape, x_true.shape)
x_true_crop = center_crop(x_true, N_crop)
plot_intensity_phase(x_true_crop)
if PTYCHOGRAPHY:
    canon = lambda x: choose_representative_broadcast(
        center_crop(P_spectral_support(x), N_crop), x_true_crop
    )
else:
    canon = lambda x: choose_representative_broadcast(
        center_crop(x, N_crop), x_true_crop
    )
lhs = reconstruction_rmse_rel(canon(x_est), center_crop(x_true, N_crop)) ** 2
bias = reconstruction_bias_rel(canon(x_est), center_crop(x_true, N_crop)) ** 2
std = reconstruction_std_rel(canon(x_est), center_crop(x_true, N_crop)) ** 2
rhs = bias + std
assert jnp.allclose(lhs, rhs, rtol=1e-6)

rel_errors = reconstruction_error_rel(canon(x_est), center_crop(x_true, N_crop)) ** 2
mse = np.mean(rel_errors)
print(mse, bias, std)

In [ ]:
for tol in [0.5, 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7]:
    # for tol in [1e-5]:
    _, spectral_support, P_spectral_support = pty_sim.construct_spectral_support_rel(
        probes_for_support[0], rel_threshold=tol
    )
    x_est = P_spectral_support(x_est_raw)
    x_true_crop = center_crop(P_spectral_support(x_true), N_crop)
    if PTYCHOGRAPHY:
        canon = lambda x: choose_representative_broadcast(
            center_crop(P_spectral_support(x), N_crop), x_true_crop
        )
    else:
        canon = lambda x: choose_representative_broadcast(
            center_crop(x, N_crop), x_true_crop
        )
    lhs = (
        reconstruction_rmse_rel(
            canon(x_est),
            canon(
                x_true,
            ),
        )
        ** 2
    )
    bias = (
        reconstruction_bias_rel(
            canon(x_est),
            canon(
                x_true,
            ),
        )
        ** 2
    )
    std = (
        reconstruction_std_rel(
            canon(x_est),
            canon(
                x_true,
            ),
        )
        ** 2
    )
    rhs = bias + std
    assert jnp.allclose(lhs, rhs, rtol=1e-6)

    rel_errors = reconstruction_error_rel(canon(x_est), canon(x_true)) ** 2
    mse = np.mean(rel_errors)
    plot_intensity_phase(
        np.mean(
            np.abs(fft2_shifted(canon(x_est)) - fft2_shifted(canon(x_true))), axis=-3
        )
    )
    plot(spectral_support)
    print(tol, mse, bias, std)

In [ ]:
from master.operators import make_crop_pad
from pty_defs import pupil_crop

full_power_x_true = jnp.sum(jnp.abs(x_true) ** 2, axis=(-2, -1))


def calc_MSE_bounds(pixel_intensity):
    meas_id = f"CDI-intensity-{pixel_intensity:.3g}"

    (
        x_true,
        d,
        probes,
        spectral_support,
        x_obj_est,
        measurement_params,
        measurement_hash_meas,
    ) = loader.get_measurement_with_probes(meas_id)
    if PTYCHOGRAPHY:
        return calc_pty_MSE_bounds_x_true_est(x_true, probes)
    else:
        return calc_cdi_MSE_bounds_x_true(x_true)


def calc_pty_MSE_bounds_x_true_est(x_true_est, probes):
    N_pupil_crop = pupil_crop(F_pupil)
    crop, pad = make_crop_pad(N_full=N_crop + 2 * N_pad, N_crop=N_pupil_crop)
    cropped_pupil = crop(F_pupil)
    ys_true = A(probes * x_true_est)
    S, Q = ptycho_SQ(probes, ys_true, pad, cropped_pupil)
    Juu, Jvv, Juv = fisher_pty_2x2_cart(S, Q)
    Jpinv_uu, Jpinv_vv, Jpinv_uv = ls_fs_cov(
        probes, ys_true, pad, x_true_est, cropped_pupil
    )[0]
    Ju_inv_2x2, Jv_inv_2x2 = diag_of_2x2_inverse(Juu, Juv, Juv, Jvv)

    # RPA: Q -> 0  =>  Juu = Jvv = 2S, Juv = 0
    Juu_rp, Jvv_rp, _ = fisher_pty_2x2_cart(S, 0.0)

    MSE_bound_2x2 = construct_MSE_bound_from_CRB(
        x_true_est, Ju_inv_2x2, Jv_inv_2x2, N_crop
    )
    MSE_bound_pinv = construct_MSE_bound_from_CRB(
        x_true_est, Jpinv_uu, Jpinv_vv, N_crop
    )
    MSE_bound_rp = construct_MSE_bound_from_CRB(
        x_true_est, 1 / Juu_rp, 1 / Jvv_rp, N_crop
    )
    return MSE_bound_2x2, MSE_bound_pinv, MSE_bound_rp


def calc_cdi_MSE_bounds_x_true(x_true):
    y_ph = fft2_shifted(x_true)
    N_y = y_ph.size

    Juu, Jvv, Juv, _, _ = fisher_2x2_block_fast_cart(x_true, y_ph, N_y)
    Ju_inv_2x2, Jv_inv_2x2 = diag_of_2x2_inverse(Juu, Juv, Juv, Jvv)
    Jpinv_uu, Jpinv_vv, Jpinv_uv = fisher_pinv_2x2_block_cart(x_true, y_ph, N_y)
    _, _, _, Ju_inv_rp, Jv_inv_rp = fisher_random_phase_cart(x_true)  # = 1/2 everywhere

    MSE_bound_2x2 = construct_MSE_bound_from_CRB(x_true, Ju_inv_2x2, Jv_inv_2x2, N_crop)
    MSE_bound_pinv = construct_MSE_bound_from_CRB(x_true, Jpinv_uu, Jpinv_vv, N_crop)
    MSE_bound_rp = construct_MSE_bound_from_CRB(x_true, Ju_inv_rp, Jv_inv_rp, N_crop)
    return MSE_bound_2x2, MSE_bound_pinv, MSE_bound_rp


MSE_bound_2x2, MSE_bound_pinv, MSE_bound_rp = calc_MSE_bounds(pixel_intensity)
print(MSE_bound_2x2, MSE_bound_pinv, MSE_bound_rp)

In [ ]:
from master.mle import (
    calc_ell_init,
    log_likelihood,
    log_likelihood_pty,
    log_likelihood_x,
)
from scipy.stats import gmean


@jit
def reconstruction_se_rel(x, x_true):
    N = x.shape[0]
    ip = lambda a, b: jnp.real(jnp.sum(jnp.conj(a) * b, axis=(-2, -1)))
    n2 = ip(x_true, x_true)
    d = x - x_true
    dbar = d.mean(0)
    c = d - dbar
    e, v, p = ip(d, d) / n2, ip(c, c) / n2, ip(dbar[None], c) / n2
    G = jnp.real(jnp.einsum("i...hw,j...hw->ij...", jnp.conj(c), c)) / n2
    trC2 = (G**2).sum((0, 1)) - (jnp.diagonal(G, axis1=0, axis2=1) ** 2).sum(-1)
    trC2 = trC2 / (N * (N - 1))
    se = lambda z: z.std(0, ddof=1) / jnp.sqrt(N)
    return dict(
        mse=se(e),
        rstd=se(v),
        rbias=jnp.sqrt(4 * p.var(0, ddof=1) / N + 2 * trC2 / N**2),
        r_eff=v.mean(0) ** 2 / trC2,
    )


def construct_estimate_mse(ell_d, x_true, x_est, ell_est):
    mse_est = dict()
    N = x_est.shape[0]
    rel_bias = reconstruction_bias_rel(x_est, x_true) ** 2
    rel_std = reconstruction_std_rel(x_est, x_true) ** 2
    mse = rel_bias + rel_std
    rel_bias_ub = rel_bias - rel_std / (N - 1)
    se = reconstruction_se_rel(x_est, x_true)
    mse_est["se_mse"] = se["mse"]
    print(x_est.shape, rel_std, se["mse"])
    mse_est["mse"] = mse
    mse_est["rbias"] = rel_bias
    mse_est["rstd"] = rel_std
    mse_est["rbias_ub"] = rel_bias_ub
    mse_est["bvr"] = rel_bias_ub / rel_std
    mse_est["se_mse"] = se["mse"]
    mse_est["se_rstd"] = se["rstd"]
    mse_est["se_rbias"] = se["rbias"]
    mse_est["se_bvr"] = jnp.sqrt(
        (se["rbias"] / rel_std) ** 2 + (mse_est["bvr"] * se["rstd"] / rel_std) ** 2
    )
    mse_est["r_eff"] = se["r_eff"]

    ell_diff = ell_d - ell_est
    norm_ell = ell_diff / ell_d
    mse_est["ell_est_gmean"] = np.asarray(gmean(ell_est))
    mse_est["ell_est_mean"] = np.nanmean(ell_est)
    mse_est["ell_gmean"] = np.asarray(gmean(norm_ell))
    mse_est["ell_mean"] = np.asarray(np.nanmean(norm_ell))
    mse_est["ell_diff_gmean"] = np.asarray(gmean(ell_diff))
    mse_est["ell_diff_mean"] = np.asarray(np.nanmean(ell_diff))
    return mse_est


def calculate_mse(pixel_intensity=1e3, test=False):
    meas_id = f"CDI-intensity-{pixel_intensity:.3g}"
    (
        x_true,
        d,
        probes,
        spectral_support,
        x_obj_est,
        measurement_params,
        measurement_hash_meas,
    ) = loader.get_measurement_with_probes(meas_id)
    if PTYCHOGRAPHY:
        x_true = P_spectral_support(x_true)
        ell_d = log_likelihood_pty(d, np.sqrt(d))
    else:
        ell_d = log_likelihood(d, np.sqrt(d))
    est_paths = loader.get_estimate_paths(meas_id, "")
    # print(est_paths)
    print(est_paths)
    mse_vals = defaultdict(dict)
    if test:
        est_paths = [est_paths[0]]
    for est_path in est_paths:
        print(f"Loading {est_path}")
        if est_path == f"cdi-128/estimates/CDI-intensity-{pixel_intensity:.3g}-HIO.h5":
            print("Skipping unaveraged HIO inits")
            continue
        x_est, simulation_params, lls, x_est_raw, config_hash, measurement_hash_est = (
            loader.get_estimate_from_path_with_x_est_raw(est_path)
        )
        assert measurement_hash_est == measurement_hash_meas
        if PTYCHOGRAPHY:
            ell_est = calc_ell_init(d, x_est, N_samples, A, probes)
        else:
            ell_est = log_likelihood_x(d, x_est)
        # plot_intensity_phase(x_true)
        # plot_intensity_phase(x_est[0,:])
        mse_est = construct_estimate_mse(
            ell_d, center_crop(x_true, N_crop), center_crop(x_est, N_crop), ell_est
        )
        mse_vals[est_path] = mse_est
        print(mse_est)
    return mse_vals

In [ ]:
vary_photon_intensity = True
if vary_photon_intensity:
    photon_intensities = [10.0**k for k in range(-1, 11)]
else:
    photon_intensities = [10.0**10]
mse_intensity_map = None
bound_dict = None
SAVE_PICKLES = True

In [ ]:
# typical runtime is 11 to 14m for all intensities for CDI
if not vary_photon_intensity or SAVE_PICKLES:
    mse_intensity_map = dict()
    for photon_intensity in photon_intensities:
        run_single_est = not vary_photon_intensity
        mse_intensity_map[photon_intensity] = calculate_mse(
            photon_intensity, run_single_est
        )

In [ ]:
if vary_photon_intensity:
    if mse_intensity_map is not None and SAVE_PICKLES:
        with open(stats_pickle_path, "wb") as handle:
            pickle.dump(mse_intensity_map, handle)
    with open(stats_pickle_path, "rb") as handle:
        mse_map = pickle.load(handle)
else:
    mse_map = mse_intensity_map

In [ ]:
# mse_map = mse_intensity_map
# for k in mse_map:
# print(mse_map[k])
# print(mse_map)

array_dict = defaultdict(lambda: defaultdict(lambda: np.zeros_like(photon_intensities)))

for i, photon_intensity in enumerate(photon_intensities):
    print(photon_intensity)
    # print(mse_map[photon_intensity])
    print()
    keys = list(mse_map[photon_intensity].keys())
    values = list(mse_map[photon_intensity].values())
    ids = list(
        enumerate(
            [
                k.split(f"CDI-intensity-{photon_intensity:.3g}-")[-1].split(".h5")[0]
                for k in keys
            ]
        )
    )
    print(ids)
    for j, id in ids:
        print(id)
        for field in [
            "mse",
            "rbias",
            "rstd",
            "ell_gmean",
            "ell_mean",
            "ell_diff_mean",
            "ell_diff_gmean",
            "se_mse",
            "se_rstd",
            "se_rbias",
            "rbias_ub",
            "bvr",
            "se_bvr",
            "r_eff",
        ]:
            print(field)
            print(values[j][field])
            array_dict[id][field][i] = values[j][field]
print(array_dict)

In [ ]:
if not vary_photon_intensity or SAVE_PICKLES:
    bound_dict = defaultdict(lambda: np.zeros_like(photon_intensities))

    for i, photon_intensity in enumerate(photon_intensities):
        MSE_bound_2x2, MSE_bound_pinv, MSE_bound_rp = calc_MSE_bounds(photon_intensity)
        bound_dict["pixel bound"][i] = MSE_bound_2x2
        bound_dict["EWCRB bound"][i] = MSE_bound_pinv
        bound_dict["RPA bound"][i] = MSE_bound_rp
    print(bound_dict)

In [ ]:
if vary_photon_intensity:
    if bound_dict is not None and SAVE_PICKLES:
        with open(bounds_pickle_path, "wb") as handle:
            pickle.dump(dict(bound_dict), handle)
    with open(bounds_pickle_path, "rb") as handle:
        bound_dict = pickle.load(handle)

In [ ]:
photon_axis = np.array(photon_intensities) * N_crop**2
PLOT_SECOND_ORDER = False
CAPSIZE = 3.0

keys = list(sorted(array_dict.keys()))
# push HIO to the back to ensure colors are consistent
if not PTYCHOGRAPHY:
    keys.remove("HIO-averaged")
    keys.append("HIO-averaged")

if PTYCHOGRAPHY:
    alpha_upper_bound = 2.5
    alpha_lower_bound = 0.01
    # alpha_lower_bound = 0.0
else:
    alpha_upper_bound = 1.25
    # alpha_lower_bound = 0.0
    alpha_lower_bound = 0.01


def plot_with_init(init="HIO"):
    fig = plt.figure(figsize=(FIG_W * 2.0, FIG_H), layout="compressed")
    print(keys)
    min_alpha = 10.0
    max_alpha = -1.0
    for k in keys:
        if init in k and (PLOT_SECOND_ORDER == ("second-order" in k)):
            if "first-order" in k:
                # TODO: consider retrieving digits lost in rounding
                alpha = float(k.split("-")[-1])
                label = rf"$\alpha = {alpha:.3g}$"
                if alpha < alpha_lower_bound or alpha > alpha_upper_bound:
                    continue
                min_alpha = min(min_alpha, alpha)
                max_alpha = max(max_alpha, alpha)
            elif k == "HIO-averaged":
                label = "HIO"
            else:
                label = k
            plt.errorbar(
                photon_axis,
                array_dict[k]["mse"],
                yerr=array_dict[k]["se_mse"],
                marker=".",
                capsize=CAPSIZE,
                linestyle="-",
                linewidth=1.0,
                label=label,
            )
    # for k in bound_dict:
    plt.plot(
        photon_axis,
        bound_dict["pixel bound"],
        color="red",
        linestyle="dashed",
        label="pixel bound",
    )
    plt.plot(
        photon_axis,
        bound_dict["RPA bound"],
        color="green",
        linestyle="dotted",
        label="RPA bound",
    )
    if not PTYCHOGRAPHY:
        plt.plot(
            photon_axis,
            bound_dict["EWCRB bound"],
            color="red",
            linestyle="dotted",
            label="EWCRB bound",
        )
    plt.legend()
    plt.yscale("log")
    plt.xscale("log")
    plt.xlim(left=0.1 * N_crop**2, right=1.1e10 * N_crop**2)
    plt.ylabel("Relative MSE")
    plt.xlabel(r"$N_{\text{photons}}$")
    plt.grid()
    if SAVE_FIGS:
        save_path = f"{fig_path}/mse_{init}-{min_alpha:g}-{max_alpha:g}.png"
    else:
        save_path = None
    if save_path is not None:
        fig.savefig(save_path, dpi=PLOT_DPI)
    plt.show()


plot_with_init(init_algo)
plot_with_init("ORACLE")

In [ ]:
def plot_bvar(init="HIO"):
    fig = plt.figure(figsize=(FIG_W * 2.0, FIG_H), layout="compressed")
    min_alpha = 10.0
    max_alpha = -1.0
    for k in keys:
        if init in k and (PLOT_SECOND_ORDER == ("second-order" in k)):
            if "first-order" in k:
                # TODO: consider retrieving digits lost in rounding
                alpha = float(k.split("-")[-1])
                label = rf"$\alpha = {alpha:.3g}$"
                if alpha < alpha_lower_bound or alpha > alpha_upper_bound:
                    continue
                min_alpha = min(min_alpha, alpha)
                max_alpha = max(max_alpha, alpha)
            elif k == "HIO-averaged":
                label = "HIO"
            else:
                label = k
            rel_bias_ub = np.maximum(
                array_dict[k]["rbias"] - array_dict[k]["rstd"] / (N_samples - 1), 0.0
            )
            lines = plt.errorbar(
                photon_axis,
                array_dict[k]["rbias"] / array_dict[k]["rstd"],
                yerr=array_dict[k]["se_bvr"],
                marker=".",
                linestyle="dashed",
                capsize=CAPSIZE,
                # errorevery=(1,2),
                # linewidth=1.0,
                # label=label,
            )
            linecolor = lines[0].get_color()
            plt.errorbar(
                photon_axis,
                rel_bias_ub / array_dict[k]["rstd"],
                yerr=array_dict[k]["se_bvr"],
                marker=".",
                color=linecolor,
                capsize=CAPSIZE,
                linestyle="-",
                # errorevery=(0,2),
                # linewidth=1.0,
                label=label,  # + " (corr.)",
            )
    plt.axhline(
        1 / N_samples, color="red", linestyle="dotted", label="Monte Carlo bias"
    )
    plt.legend()
    if init == "ORACLE":
        plt.yscale("symlog", linthresh=0.001)
    else:
        plt.yscale("log")
    plt.xscale("log")
    plt.ylabel("BVR")
    plt.xlabel(r"$N_{\text{photons}}$")
    plt.xlim(left=0.1 * N_crop**2, right=1.1e10 * N_crop**2)
    plt.grid()
    if SAVE_FIGS:
        save_path = f"{fig_path}/bvr_{init}-{min_alpha:g}-{max_alpha:g}.png"
    else:
        save_path = None
    if save_path is not None:
        fig.savefig(save_path, dpi=PLOT_DPI)
    plt.show()


plot_bvar(init_algo)
plot_bvar("ORACLE")

In [ ]:
photon_axis = np.array(photon_intensities) * N_crop**2

keys = list(sorted(array_dict.keys()))
# push HIO to the back to ensure colors are consistent
if not PTYCHOGRAPHY:
    keys.remove("HIO-averaged")
    keys.append("HIO-averaged")


def plot_ell(init="HIO"):
    fig = plt.figure(figsize=(FIG_W * 2.0, FIG_H), layout="compressed")
    for k in keys:
        if init in k and "second-order" not in k:
            if "first-order" in k:
                # TODO: consider retrieving digits lost in rounding
                alpha = float(k.split("-")[-1])
                label = rf"$\alpha = {alpha:.3g}$"
                if alpha < alpha_lower_bound or alpha > alpha_upper_bound:
                    continue
            elif k == "HIO-averaged":
                label = "HIO"
            else:
                label = k
            plt.plot(
                photon_axis,
                2 * array_dict[k]["ell_mean"],
                marker=".",
                linestyle="-",
                linewidth=1.0,
                label=label,
            )
    # for k in bound_dict:
    # plt.plot(photon_axis, bound_dict["pixel bound"], color="red",linestyle="dashed", label="pixel bound")
    # plt.plot(photon_axis, bound_dict["EWCRB bound"], color="red",linestyle="dotted", label="EWCRB bound")
    plt.legend()
    plt.yscale("log")
    plt.xscale("log")
    plt.xlim(left=0.1 * N_crop**2, right=1.1e10 * N_crop**2)
    plt.ylabel(
        r"$\frac{\ell(d; \sqrt{d}) - \ell(d; A\hat{x}_{final})}{\ell(d; \sqrt{d})}$"
    )
    plt.xlabel(r"$N_{\text{photons}}$")
    plt.grid()
    if SAVE_FIGS:
        save_path = f"{fig_path}/ell_{init}.png"
    else:
        save_path = None
    if save_path is not None:
        fig.savefig(save_path, dpi=PLOT_DPI)
    plt.show()


plot_ell(init_algo)
plot_ell("ORACLE")